# Blick: pipeline de segmentação e classificação de plantas de milho

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blick-ai/blick-treino/blob/main/notebooks/pipeline_milho_executavel.ipynb)

Este notebook roda **sem Google Drive**: ele baixa o repositório (pesos e fotos de exemplo) e executa o pipeline completo.

**Como usar:** abra no Colab e clique em *Ambiente de execução > Executar tudo*. Funciona em CPU; com GPU (T4) fica mais rápido.

**Pipeline (duas redes neurais convolucionais, família YOLO26):**
1. **Segmentação de instâncias** (`yolo26n-seg`): acha cada planta de milho e devolve um polígono.
2. **Isolamento**: recorta cada planta e pinta de preto o que está fora do polígono.
3. **Classificação** (`yolo26n-cls`): classifica cada recorte como `healthy` ou `unhealthy`.

**Regra por imagem:** basta uma planta `unhealthy` para a foto inteira ser `unhealthy`.

## 1. Instalação e download do repositório

In [ ]:
!pip install -q ultralytics

import os
REPO_URL = "https://github.com/blick-ai/blick-treino.git"
if not os.path.exists("blick-treino"):
    !git clone -q {REPO_URL}
%cd blick-treino
!ls

## 2. Carrega os dois modelos

In [ ]:
import torch
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Dispositivo:", "GPU" if DEVICE == 0 else "CPU")

model_seg = YOLO("weights/segmentacao_best.pt")
model_cls = YOLO("weights/classificacao_best.pt")
print("Classes da segmentação:", model_seg.names)
print("Classes da classificação:", model_cls.names)

# O modelo de segmentação tem duas classes: a 0 ('Blick-corn-analyzer') não tem anotações,
# e a 1 ('corn') é o milho. A busca precisa ser por nome EXATO, não por substring.
IDX_CLASSE_CORN = next(i for i, n in model_seg.names.items() if n.lower() in ("corn", "milho"))
print("Índice da classe corn:", IDX_CLASSE_CORN)

## 3. Fotos de exemplo

O notebook usa até 4 fotos da pasta `exemplos/` do repositório. Se não houver fotos lá, aparece um botão para você enviar as suas.

In [ ]:
from pathlib import Path
EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
MAX_IMAGENS = 4

pasta = Path("exemplos/entrada")
imagens = sorted(p for p in pasta.glob("*") if p.suffix.lower() in EXT)[:MAX_IMAGENS] if pasta.exists() else []

if not imagens:
    print("Nenhuma foto em exemplos/entrada. Envie até 4 fotos de milho:")
    from google.colab import files
    pasta.mkdir(parents=True, exist_ok=True)
    for nome, conteudo in files.upload().items():
        (pasta / nome).write_bytes(conteudo)
    imagens = sorted(p for p in pasta.glob("*") if p.suffix.lower() in EXT)[:MAX_IMAGENS]

print(f"{len(imagens)} imagem(ns):")
for p in imagens:
    print(" ", p.name)

## 4. Funções do pipeline

`CONF_SEG` é o limiar de confiança do segmentador. As métricas do artigo foram medidas com 0,40; a API opera com 0,25. Para fotos escuras ou com plantas distantes, vale testar 0,25.

In [ ]:
import cv2
import numpy as np

CONF_SEG = 0.40
PADDING = 15


def isolar_instancias(imagem, poligonos, padding=PADDING):
    altura, largura = imagem.shape[:2]
    saida = []
    for poligono in poligonos:
        pontos = np.round(poligono).astype(np.int32)
        if len(pontos) < 3:
            continue
        mascara = np.zeros((altura, largura), dtype=np.uint8)
        cv2.fillPoly(mascara, [pontos], 255)
        isolado = cv2.bitwise_and(imagem, imagem, mask=mascara)
        x, y, w, h = cv2.boundingRect(mascara)
        x1, y1 = max(0, x - padding), max(0, y - padding)
        x2, y2 = min(largura, x + w + padding), min(altura, y + h + padding)
        crop = isolado[y1:y2, x1:x2]
        if crop.size == 0:
            continue
        saida.append({"crop": crop, "bbox": (x1, y1, x2, y2), "poligono": pontos})
    return saida


def processar_imagem(caminho, conf_seg=CONF_SEG):
    imagem = cv2.imread(str(caminho))
    if imagem is None:
        raise IOError(f"Não consegui abrir {caminho}")

    res = model_seg.predict(imagem, conf=conf_seg, device=DEVICE, verbose=False)[0]
    polis = []
    if res.masks is not None:
        classes = res.boxes.cls.cpu().numpy().astype(int)
        confs = res.boxes.conf.cpu().numpy()
        for poli, c, conf in zip(res.masks.xy, classes, confs):
            if c == IDX_CLASSE_CORN:
                polis.append((poli, float(conf)))

    instancias = []
    if polis:
        isolados = isolar_instancias(imagem, [p for p, _ in polis])
        for iso, (_, conf_det) in zip(isolados, polis):
            r = model_cls.predict(iso["crop"], device=DEVICE, verbose=False)[0]
            instancias.append({
                "poligono": iso["poligono"], "bbox": iso["bbox"],
                "conf_deteccao": conf_det,
                "classe_saude": r.names[r.probs.top1],
                "conf_saude": float(r.probs.top1conf),
            })

    if not instancias:
        veredito = "sem milho detectado"
    elif any(i["classe_saude"] == "unhealthy" for i in instancias):
        veredito = "unhealthy"
    else:
        veredito = "healthy"
    return {"caminho": str(caminho), "imagem": imagem, "instancias": instancias, "veredito": veredito}

## 5. Roda o pipeline

In [ ]:
resultados = [processar_imagem(p) for p in imagens]

for r in resultados:
    nh = sum(i["classe_saude"] == "healthy" for i in r["instancias"])
    nu = sum(i["classe_saude"] == "unhealthy" for i in r["instancias"])
    print(f"{Path(r['caminho']).name:40s} -> {len(r['instancias'])} planta(s) ({nh} healthy, {nu} unhealthy) | veredito: {r['veredito'].upper()}")

## 6. Visualização

Verde = `healthy`, vermelho = `unhealthy`. O título mostra o veredito da imagem.

In [ ]:
import matplotlib.pyplot as plt

COR = {"healthy": (0, 200, 0), "unhealthy": (220, 0, 0)}


def desenhar(r):
    img = cv2.cvtColor(r["imagem"], cv2.COLOR_BGR2RGB).copy()
    for i in r["instancias"]:
        cor = COR[i["classe_saude"]]
        cv2.polylines(img, [i["poligono"]], True, cor, 3)
        x1, y1, _, _ = i["bbox"]
        cv2.putText(img, f"{i['classe_saude']} {i['conf_saude']:.0%}", (x1, max(0, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, cor, 2, cv2.LINE_AA)
    return img


n_col = min(2, max(1, len(resultados)))
n_lin = (len(resultados) + n_col - 1) // n_col
fig, axes = plt.subplots(n_lin, n_col, figsize=(7 * n_col, 5 * n_lin))
axes = np.array(axes).reshape(-1)
for ax, r in zip(axes, resultados):
    ax.imshow(desenhar(r))
    cor_t = {"unhealthy": "red", "healthy": "green"}.get(r["veredito"], "gray")
    ax.set_title(f"{Path(r['caminho']).name}\nveredito: {r['veredito'].upper()}", color=cor_t, fontsize=10)
    ax.axis("off")
for ax in axes[len(resultados):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Resultados do treinamento

Curvas de loss (treino contra validação) e matrizes de confusão no conjunto de teste, dos dois modelos.

In [ ]:
import matplotlib.image as mpimg

for titulo, caminho in [
    ("Segmentação: loss de treino e validação", "resultados/segmentacao/loss_treino_validacao_seg.png"),
    ("Classificação: loss de treino e validação", "resultados/classificacao/loss_treino_validacao.png"),
]:
    if os.path.exists(caminho):
        plt.figure(figsize=(12, 4))
        plt.imshow(mpimg.imread(caminho)); plt.axis("off"); plt.title(titulo)
        plt.show()
    else:
        print("Arquivo não encontrado:", caminho)

In [ ]:
# Matrizes de confusão no conjunto de teste (valores dos relatórios de treino)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Segmentação (conf=0.4): linhas = previsto, colunas = real
cm_seg = np.array([[2687, 303], [1108, 0]])
axes[0].imshow(cm_seg, cmap="Blues")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["corn", "background"])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(["corn", "background"])
axes[0].set_xlabel("Real"); axes[0].set_ylabel("Previsto")
axes[0].set_title("Segmentação: teste, conf=0.4")
for i in range(2):
    for j in range(2):
        if (i, j) != (1, 1):
            axes[0].text(j, i, str(cm_seg[i, j]), ha="center", va="center",
                         color="white" if cm_seg[i, j] > 1500 else "black", fontsize=13)

# Classificação: linhas = real, colunas = previsto
cm_cls = np.array([[2242, 25], [39, 211]])
axes[1].imshow(cm_cls, cmap="Blues")
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(["healthy", "unhealthy"])
axes[1].set_yticks([0, 1]); axes[1].set_yticklabels(["healthy", "unhealthy"])
axes[1].set_xlabel("Previsto"); axes[1].set_ylabel("Real")
axes[1].set_title("Classificação: teste")
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, str(cm_cls[i, j]), ha="center", va="center",
                     color="white" if cm_cls[i, j] > 1000 else "black", fontsize=13)

plt.tight_layout()
plt.show()

## 8. Treinamento (opcional)

As células abaixo **treinam os dois modelos do zero**, com os mesmos hiperparâmetros do TCC. Elas ficam desligadas por padrão (`RETREINAR = False`) porque:

- precisam do **dataset**, que não está no repositório (tamanho e licença das imagens);
- precisam de **GPU** (Ambiente de execução > Alterar tipo de ambiente > T4);
- levam muito tempo com as 100 épocas originais (cerca de 1h30 a 2h por modelo). Para uma demonstração rápida, use `EPOCAS = 3`.

Para ligar: coloque o link de download do dataset em `URL_DATASET_CLS` e/ou `URL_DATASET_SEG` (link compartilhado do Google Drive, "qualquer pessoa com o link") e mude `RETREINAR` para `True`. O treino novo é salvo em `runs/` e **não** sobrescreve os pesos de `weights/`.

In [ ]:
RETREINAR = False          # mude para True para treinar
EPOCAS = 100               # use 3 para uma demonstração rápida (os resultados NÃO serão equivalentes)
URL_DATASET_CLS = ""       # zip do dataset de classificação (formato Folder Structure: train/valid/test, cada uma com healthy/ e unhealthy/)
URL_DATASET_SEG = ""       # zip do dataset de segmentação (formato COCO Segmentation exportado do Roboflow)

In [ ]:
# --- Treino do CLASSIFICADOR (yolo26n-cls): 224x224, lote 64, paciência 20 ---
if RETREINAR and URL_DATASET_CLS:
    !pip install -q gdown
    import gdown, zipfile
    gdown.download(URL_DATASET_CLS, "dataset_cls.zip", quiet=False, fuzzy=True)
    zipfile.ZipFile("dataset_cls.zip").extractall("dataset_cls")

    novo_cls = YOLO("yolo26n-cls.pt")
    novo_cls.train(
        data="dataset_cls",
        epochs=EPOCAS,
        imgsz=224,
        batch=64,
        patience=20,
        name="milho_healthy_unhealthy_retreino",
        device=0,
        cache="ram",
    )
    print("Pesos salvos em runs/classify/milho_healthy_unhealthy_retreino/weights/best.pt")
else:
    print("Treino do classificador desligado (RETREINAR=False ou URL_DATASET_CLS vazia).")

In [ ]:
# --- Treino do SEGMENTADOR (yolo26n-seg): 640x640, lote 16, paciência 20 ---
if RETREINAR and URL_DATASET_SEG:
    !pip install -q gdown pycocotools
    import gdown, zipfile, glob, shutil, json as json_lib
    from pycocotools import mask as mask_utils

    gdown.download(URL_DATASET_SEG, "dataset_seg_coco.zip", quiet=False, fuzzy=True)
    zipfile.ZipFile("dataset_seg_coco.zip").extractall("coco_raw")

    def json_do_split(padrao):
        achados = [j for j in glob.glob("coco_raw/**/*.json", recursive=True)
                   if padrao in os.path.basename(os.path.dirname(j)).lower()]
        assert len(achados) == 1, f"Esperava 1 json para '{padrao}', achei {achados}"
        return achados[0]

    splits_json = {"train": json_do_split("train"), "valid": json_do_split("val"), "test": json_do_split("test")}

    def poligonos_da_anotacao(seg, w, h):
        polis = []
        if isinstance(seg, dict):  # máscara RLE -> contorno
            rle = mask_utils.frPyObjects(seg, h, w) if isinstance(seg.get("counts"), list) else seg
            m = mask_utils.decode(rle)
            m = m[:, :, 0] if m.ndim == 3 else m
            cont, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            polis += [c.reshape(-1, 2).astype(float) for c in cont if len(c) >= 3]
        elif isinstance(seg, list):
            polis += [np.array(p, dtype=float).reshape(-1, 2) for p in seg if isinstance(p, list) and len(p) >= 6]
        return polis

    categorias = sorted(json_lib.load(open(splits_json["train"]))["categories"], key=lambda c: c["id"])
    cat_idx = {c["id"]: i for i, c in enumerate(categorias)}
    nomes = [c["name"] for c in categorias]

    for split, jpath in splits_json.items():
        d = json_lib.load(open(jpath))
        pasta = Path(jpath).parent
        out_i, out_l = Path("dataset_seg/images")/split, Path("dataset_seg/labels")/split
        out_i.mkdir(parents=True, exist_ok=True); out_l.mkdir(parents=True, exist_ok=True)
        anns = {}
        for a in d["annotations"]:
            anns.setdefault(a["image_id"], []).append(a)
        for img in d["images"]:
            if not (pasta / img["file_name"]).exists():
                continue
            shutil.copy(pasta / img["file_name"], out_i / img["file_name"])
            linhas = []
            for a in anns.get(img["id"], []):
                for pts in poligonos_da_anotacao(a.get("segmentation"), img["width"], img["height"]):
                    coords = [f"{v:.6f}" for x, y in pts for v in (x / img["width"], y / img["height"])]
                    linhas.append(str(cat_idx[a["category_id"]]) + " " + " ".join(coords))
            if linhas:   # imagens sem anotação (sem milho) ficam sem .txt
                (out_l / (Path(img["file_name"]).stem + ".txt")).write_text("\n".join(linhas))

    yaml = "path: " + str(Path("dataset_seg").resolve()) + "\ntrain: images/train\nval: images/valid\ntest: images/test\nnames:\n"
    yaml += "".join(f"  {i}: {n}\n" for i, n in enumerate(nomes))
    Path("dataset_seg/data.yaml").write_text(yaml)

    novo_seg = YOLO("yolo26n-seg.pt")
    novo_seg.train(
        data="dataset_seg/data.yaml",
        epochs=EPOCAS,
        imgsz=640,
        batch=16,
        patience=20,
        name="milho_corn_seg_retreino",
        device=0,
    )
    print("Pesos salvos em runs/segment/milho_corn_seg_retreino/weights/best.pt")
else:
    print("Treino do segmentador desligado (RETREINAR=False ou URL_DATASET_SEG vazia).")

## Limitações

- O rótulo `unhealthy` corresponde à presença de um adesivo simulando sintoma; o reconhecimento de doenças reais não foi testado.
- A divisão treino/validação/teste do classificador foi estratificada por classe e feita por recorte, não por foto de origem, então as métricas do classificador são otimistas.
- O segmentador omite cerca de um terço das plantas (recall de máscara 0,67), e isso limita o que chega ao classificador.